In [ ]:
import requests
from urllib.parse import unquote

BASE = "https://apiv2-observatorio.sebrae.com.br/tesseract/data.jsonrecords"
MUN = "3550308"  # São Paulo

queries = [
    # Empresas por setor e clientes Sebrae
    f"{BASE}?Registration+Status=2&Sebrae+Target+Audience+Indicator=1&Grande+Sector=1,2,3,4&cube=RF&drilldowns=Sebrae+Client+Indicator,Grande+Sector&locale=pt&measures=Establishments&Municipality={MUN}",
    # Empresas por natureza jurídica
    f"{BASE}?Municipality={MUN}&Registration+Status=2&cube=RF&drilldowns=Legal+Nature&measures=Establishments&parents=true&exclude=Type+Legal+Nature:0",
    # Evolução de novas empresas por ano e porte
    f"{BASE}?Municipality={MUN}&Registration+Status=2&cube=RF&drilldowns=Open+Activity+Year,Company+Size+Sebrae&measures=Establishments&time=Open+Activity+Year.gte.1980&exclude=Company+Size+Sebrae:0",
    # Empresas por porte e setor (matriz para SAM)
    f"{BASE}?Municipality={MUN}&Registration+Status=2&cube=RF&drilldowns=Grande+Sector,Company+Size+Sebrae&measures=Establishments&exclude=Company+Size+Sebrae:0",
    # CAGED - movimentações por setor 2026
    f"{BASE}?Municipality={MUN}&Grande+Sector=1,2,3,4,5&Year=2026&cube=CAGED_movements&drilldowns=Grande%20Sector&measures=Movement+Balance,Admissions,Resignations",
    # RAIS - empregados e remuneração por setor/ano (MPE)
    f"{BASE}?Municipality={MUN}&Aggregated+Company+Size=1&Commercial+Companies+Indicator=1&cube=RAIS_workers&drilldowns=Grande%20Sector,Year&measures=Remuneration+Avg+Nominal,Workers&Active+worker+indicator=1",
    # RAIS - distribuição por subgrupo 2023 vs 2024
    f"{BASE}?Active+worker+indicator=1&Commercial+Companies+Indicator=1&Aggregated+Company+Size=1&Municipality={MUN}&Year=2023,2024&cube=RAIS_workers&drilldowns=Year,Subgroup,Active+worker+indicator&measures=Workers,Remuneration+Avg+Nominal&parents=true&growth=Year.Workers.fixed.2023",
    # IBGE - pirâmide etária (Censo 2022)
    f"{BASE}?Municipality={MUN}&Year=2022&cube=IBGE_Censo_Sexo_Faixa_Etaria&drilldowns=Age+Group,Sex&locale=pt&measures=Population",
    # IBGE - evolução populacional município
    f"{BASE}?Municipality={MUN}&cube=IBGE_Censo_Pop_Sit&drilldowns=Year,Municipality&locale=pt&measures=Population",
    # PNUD - Índice Gini município
    f"{BASE}?Year=2000,2010&Municipality={MUN}&cube=PNUD_Atlas_IDHM&drilldowns=Year,Municipality&locale=pt&measures=Gini",
    # INEP - distorção idade-série
    f"{BASE}?Education+Level=0,3&Municipality={MUN}&cube=INEP_Censo_Taxas_Basica&drilldowns=Year,Education+Level&locale=pt&measures=Age-Grade+Distortion+Rate",
    # INEP - abandono escolar
    f"{BASE}?Education+Level=0,3&Municipality={MUN}&cube=INEP_Censo_Taxas_Basica&drilldowns=Year%2CEducation+Level&locale=pt&measures=Dropout+Rate",
]


def get_param(url, name):
    """Extrai e decodifica um parâmetro da query string."""
    for part in url.split("?", 1)[1].split("&"):
        if part.startswith(name + "="):
            return unquote(part.split("=", 1)[1].replace("+", " "))
    return ""


all_data = {}
for url in queries:
    r = requests.get(url)
    data = r.json()
    cube = get_param(url, "cube")
    measures = get_param(url, "measures")
    drilldowns = get_param(url, "drilldowns")
    # Chave única: cube + measures + drilldowns evita colisão entre queries do mesmo cubo
    key = f"{cube}__{measures}__{drilldowns}"
    all_data[key] = data
    print(f"✅ {key}: {data['page']['total']} registros")

In [3]:
import requests
cubes = requests.get("https://apiv2-observatorio.sebrae.com.br/tesseract/cubes").json()
for cube in cubes["cubes"]:
    print(cube["name"], "—", cube["annotations"].get("table", ""))

INEP_Censo_Ed_Basica — Censo da Educação Básica
PCI — 
IBGE_Censo_Pop_Sit — Pop por Situação Domiciliar
DATASUS_Estimativas_Populacionais — Estimativas Populacionais
credito_bacen_valores — Credito
Relatedness — 
Sebrae_Atendimento — Atendimentos
PPM Efetivo dos Rebanhos — PPM - Efetivo dos Rebanhos
Digital_Market_Comparation — Comparativo de Pesquisas
IBGE_Censo_Densidade_Demografica — Densidade demográfica
IBGE_PIB_Municipal_VAB — PIB Municipal - Valor Adicionado Bruto
ANATEL_IBC_MUN — IBC MUN - Normalizado
bcb_sicor_prod — SICOR - Valores Contratos por Produto
radar_predictions — Previsões
PEVS_Plant_Extraction — Extração Vegetal
MTUR_Mapa_Turismo — Mapa do Turismo
SICONFI_receitas — Receitas
PEVS_Area_Forestry — Área Total Silvicultura
INEP_enem — ENEM
rf_notas_fiscais — Notas Fiscais
INEP_Censo_Taxas_Basica — Taxas - Educação Básica
CAGED_movements — Movimentação
MTUR_Chegadas_Internacionais — Chegadas Internacionais
BCB_AGENCIAS_POSTOS — BCB - Agências e Postos
PNUD_Atlas_IDHM 

In [6]:
with open("../data/raw/sebrae.json", "w", encoding="utf-8") as f:
    import json
    json.dump(all_data, f, indent=2, ensure_ascii=False)